## Finance Researcher with TODO Planner

Links
- https://github.com/laxmimerit/MCP-Mastery-with-Claude-and-Langchain
- https://aistudio.google.com/api-keys
- https://pypi.org/project/fastmcp/
- https://pypi.org/project/yfinance/

In [23]:
import warnings 
warnings.filterwarnings("ignore")

In [24]:
from dotenv import load_dotenv
load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent, AgentState
from langchain_core.tools import tool, InjectedToolCallId
from langchain_core.messages import HumanMessage, ToolMessage
from langgraph.prebuilt import InjectedState
from langgraph.types import Command

from typing import Annotated, Literal, NotRequired
from typing_extensions import TypedDict

import subprocess
import sys

### Yahoo Finance Research Tool

In [25]:
@tool
def finance_research(query):
    """Research stocks using Yahoo Finance MCP async function."""

    code = f"""
import asyncio
from yahoo_mcp import finance_research
asyncio.run(finance_research("{query}"))
"""
    
    result = subprocess.run([sys.executable, '-c', code], capture_output=True, text=True)

    return result.stdout

In [26]:
query = "What is the current stock price of Apple (AAPL)?"
response = finance_research.invoke({"query": query})

print(response)

Loaded 9 Tools
Tools Available: ['get_historical_stock_prices', 'get_stock_info', 'get_yahoo_finance_news', 'get_stock_actions', 'get_financial_statement', 'get_holder_info', 'get_option_expiration_dates', 'get_option_chain', 'get_recommendations']
The current stock price of Apple (AAPL) is $276.97.



### TODO Tools

In [27]:
from typing import Literal
from typing_extensions import TypedDict

# Define Todo structure
class Todo(TypedDict):
    """A task item for tracking progress."""
    content: str
    status: Literal["pending", "in_progress", "completed"]

# Define custom state schema with todos
class DeepAgentState(AgentState):
    """Extended agent state that includes TODO tracking."""
    todos: NotRequired[list[Todo]]

In [28]:
@tool
def write_todos(
    todos: list[Todo], 
    tool_call_id: Annotated[str, InjectedToolCallId]
) -> Command:
    """Create or update the TODO list for task planning and tracking.
    
    Args:
        todos: List of Todo items with content and status
    
    Returns:
        Command to update agent state with new TODO list
    """
    result = "Updated TODO List:\n"
    for i, todo in enumerate(todos, 1):
        status_icon = {"pending": "[ ]", "in_progress": "[~]", "completed": "[x]"}
        icon = status_icon.get(todo["status"], "[ ]")
        result += f"{i}. {icon} {todo['content']} ({todo['status']})\n"
    
    return Command(
        update={
            "todos": todos,
            "messages": [ToolMessage(result, tool_call_id=tool_call_id)]
        }
    )

In [29]:
@tool
def read_todos(
    state: Annotated[DeepAgentState, InjectedState],
    tool_call_id: Annotated[str, InjectedToolCallId]
) -> str:
    """Read the current TODO list to remind yourself of the plan.
    
    Returns:
        The current TODO list with status indicators
    """
    todos = state.get("todos", [])
    if not todos:
        return "No todos currently in the list."
    
    result = "Current TODO List:\n"
    for i, todo in enumerate(todos, 1):
        status_icon = {"pending": "[ ]", "in_progress": "[~]", "completed": "[x]"}
        icon = status_icon.get(todo["status"], "[ ]")
        result += f"{i}. {icon} {todo['content']} ({todo['status']})\n"
    
    return result.strip()

In [30]:
# Test the TODO tools
sample_todos = [
    {"content": "Get AAPL stock info", "status": "completed"},
    {"content": "Get AAPL news", "status": "in_progress"},
    {"content": "Analyze AAPL financials", "status": "pending"}
]

### Finance Research Agent with TODO Planner

In [31]:
# Initialize the LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

In [32]:
# Combine all tools
tools = [finance_research, write_todos, read_todos]

In [33]:
system_prompt = """You are a financial research analyst with access to:
- finance_research(query): Research stocks using Yahoo Finance data
- write_todos(todos): Create and update TODO list for tracking progress
- read_todos(): Read current TODO list

Goal: Conduct thorough financial research using TODO planning for complex tasks.

### Workflow for Complex Research Tasks
1. Create a TODO list using write_todos with specific research tasks
2. Execute each task using finance_research tool
3. After completing each task, update the TODO list (mark as completed, move to next)
4. Use read_todos to remind yourself of the plan
5. Continue until all tasks are completed
6. Provide final comprehensive analysis

### When to Use TODO List
- Multi-step research (analyzing multiple stocks, comparing companies, deep analysis)
- Complex queries requiring multiple data points
- Any task with 3+ research steps

### For Simple Queries
- Single stock price check → just use finance_research directly
- Quick news lookup → just use finance_research directly

### Rules
- Always use finance_research tool to get actual data (don't make up numbers)
- Keep TODO tasks specific and actionable
- Update TODO status as you progress (mark completed tasks as "completed")
- After completing a task, call read_todos to remind yourself of remaining tasks
- Provide clear, data-driven insights

IMPORTANT: Always create a TODO list for complex research requests and update it as you progress.
"""

In [34]:
# Create the finance research agent with state_schema
agent = create_agent(
    model=llm, 
    tools=tools, 
    system_prompt=system_prompt,
    state_schema=DeepAgentState
)

In [35]:
from langchain_core.messages import AIMessage

def stream_agent(query):
    """Stream agent execution and print tool calls and messages."""
    
    query = {'messages': [HumanMessage(content=query)]}
    
    for chunk in agent.stream(query, stream_mode="messages"):
        message = chunk[0]
        
        # Print AI messages with tool calls
        if isinstance(message, AIMessage):
            if message.tool_calls:
                print("\n[AI Tool Calls]")
                for tool_call in message.tool_calls:
                    print(f"  Tool: {tool_call['name']}")
                    print(f"  Args: {tool_call['args']}")
        
        # Print ToolMessage responses
        elif isinstance(message, ToolMessage):
            print(f"\n[Tool Response: {message.name}]")
            content = message.content
            if len(content) > 500:
                print(f"{content[:500]}...")
            else:
                print(content)

### Example: Complex Research with TODO Planning

In [36]:
query = """Perform a comprehensive analysis of Apple (AAPL):
- Current stock price and performance
- Latest news
- Financial statements
- Analyst recommendations"""

stream_agent(query)


[AI Tool Calls]
  Tool: write_todos
  Args: {'todos': [{'status': 'pending', 'content': 'Get current stock price and performance for AAPL'}, {'status': 'pending', 'content': 'Get latest news for AAPL'}, {'status': 'pending', 'content': 'Get financial statements for AAPL'}, {'status': 'pending', 'content': 'Get analyst recommendations for AAPL'}]}

[Tool Response: write_todos]
Updated TODO List:
1. [ ] Get current stock price and performance for AAPL (pending)
2. [ ] Get latest news for AAPL (pending)
3. [ ] Get financial statements for AAPL (pending)
4. [ ] Get analyst recommendations for AAPL (pending)


[AI Tool Calls]
  Tool: finance_research
  Args: {'query': 'AAPL stock price and performance'}

[Tool Response: finance_research]
Loaded 9 Tools
Tools Available: ['get_historical_stock_prices', 'get_stock_info', 'get_yahoo_finance_news', 'get_stock_actions', 'get_financial_statement', 'get_holder_info', 'get_option_expiration_dates', 'get_option_chain', 'get_recommendations']
Here's 